In [0]:
%sql
CREATE OR REPLACE TABLE workspace.property.silver_listings AS
WITH cleaned AS (
  SELECT
    listing_id,
    initcap(trim(suburb)) AS suburb,
    city,
    property_type,
    CASE WHEN lower(trim(CAST(bedrooms AS STRING))) = 'studio' THEN 0
         ELSE try_cast(trim(CAST(bedrooms AS STRING)) AS INT) END AS bedrooms,
    try_cast(bathrooms AS INT) AS bathrooms,
    try_cast(floor_size_sqm AS INT) AS floor_size_sqm,
    try_cast(erf_size_sqm AS INT) AS erf_size_sqm,
    try_cast(regexp_replace(CAST(asking_price AS STRING), '[^0-9]', '') AS BIGINT) AS asking_price,
    to_date(listing_date) AS listing_date,
    initcap(trim(status)) AS status,
    agent
  FROM workspace.property.bronze_listings
),
deduped AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY listing_id ORDER BY listing_date) AS rn
  FROM cleaned
)
SELECT
  listing_id, suburb, city, property_type, bedrooms, bathrooms,
  floor_size_sqm, erf_size_sqm,
  CASE WHEN asking_price BETWEEN 100000 AND 100000000 THEN asking_price END AS asking_price,
  listing_date, status, agent
FROM deduped
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT suburb) AS distinct_suburbs,
       SUM(CASE WHEN asking_price IS NULL THEN 1 ELSE 0 END) AS missing_prices
FROM workspace.property.silver_listings;

total_rows,distinct_suburbs,missing_prices
600,16,40
